In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M15.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7775887570957036, 'n_it': 0.39314807131607676}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 1000

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.PseudorandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[13.595852279640498, 15.60821863184435, 18.088045694296717, 13.69196450226673, 15.19063204074173, 15.352671505921174, 13.870875422771741, 15.139508806572431, 17.887848963550645, 13.759892181096504, 14.89682889863797, 16.79070726593906, 15.787122681389022, 14.38743741883001, 14.71045074568343, 13.73517888651283, 14.346785112244556, 17.46894782933698, 13.669100804771519, 13.99730438793827, 14.7955995853536, 17.78706461829102, 16.23200994031452, 13.75430430298875, 13.871690140184068, 14.804022809124662, 16.429934020497143, 13.958280073785758, 15.899031661695625, 16.006493647490984, 15.526108042843044, 14.295869212173374, 17.590711995926537, 13.721932716713583, 13.525817186816242, 14.42243794379689, 15.632286938051514, 15.55222791330863, 14.257395634242027, 15.353401188080838, 14.98313895267385, 16.82002144520788, 16.14694474783333, 15.49273211094552, 13.437215673627104, 14.803433269061827, 17.43467319113953, 14.87450131387419, 16.66329169745072, 13.662130694999608, 16.383404800890954, 14.

In [5]:
np.average(y_max_arr)

np.float64(14.993586005068462)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M15/DataGenerated/pseudorandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)